In [336]:
from bs4 import BeautifulSoup
import copy
import json
import os
from PIL import Image

In [367]:
def determine_page_from_x_coordinate(images, x):
    offset = 0
    for page_idx, page in enumerate(images[::-1]):  # pages are read from right to left
        offset += image.width
        if x < offset:
            return len(images)-1-page_idx
        
def segment_into_blocks(boxes):
    """
    Group boxes of same box type together. We need this so we can choose the boundaries of the
    TEI components correctly, e.g. <head> enclosing the title boxes, etc.

    Also, the music boxes contain two positions each: The right side contains the musical notation,
    while the left side contains the lyric character. To correctly encode this, we need to split the
    data accordingly.

    For this, we need column break information to determine the position of each box correctly!
    """
    boxes = copy.deepcopy(boxes)
    blocks = []
    for box in boxes:
        # i)   If there are no blocks yet, make a new block.
        # ii)  If there is a block but the current box' type is different
        #      than the current block's type, make a new block.
        if len(blocks) and blocks[-1][-1]["box_type"] == box["box_type"]:
            blocks[-1].append(box)
        else:
            blocks.append([box])
    return blocks
    
def add_block_tags(blocks):
    blocks = copy.deepcopy(blocks)
    for block in blocks:
        block_type = block[0]["box_type"]
        if block_type == "Title":  # titles are <head>
            block[0]["text_content"] = "<head>" + block[0]["text_content"]
            block[-1]["text_content"] += "</head>"
        elif block_type == "Preface":
            block[0]["text_content"] = '<p ana="preface">' + block[0]["text_content"]
            block[-1]["text_content"] += "</p>"
        elif block_type == "Mode":
            block[0]["text_content"] = '<p ana="mode" rend="small">' + block[0]["text_content"]
            block[-1]["text_content"] += "</p>"
        elif block_type == "Music":
            block[0]["text_content"] = '<p ana="lyrics"><lg><l>' + block[0]["text_content"]
            block[-1]["text_content"] += "</l></lg></p>"
    return blocks

def resolve_blocks_to_boxes(blocks):
    boxes = []
    for block in blocks:
        block_type = block[0]["box_type"]
        if block_type == "Music":
            if False: #block[0]["notation_content"] is not None:
                for box in block:
                    notation_box = copy.deepcopy(box)
                    notation_box["type"] = "NOTATION"
                    if box["notation_content"] is None:  # stanza separation => line groups
                        notation_box["tei"] = "</l> <l>"
                    else:
                        notation_box["tei"] = notation_box["text_content"]
                    notation_box["coordinates"] = notation_box["notation_coordinates"]
                    boxes.append(notation_box)
                
            for box in block:
                lyric_box = box
                lyric_box["type"] = "LYRIC"
                if box["text_content"] == "":  # stanza separation => line groups
                    lyric_box["tei"] = "</l></lg> <lg><l>"
                else:
                    lyric_box["tei"] = lyric_box["text_content"]
                lyric_box["coordinates"] = lyric_box["text_coordinates"]
                if lyric_box["tei"] is not None:  # do not consider music characters without any text content
                    boxes.append(lyric_box)
        else:
            for box in block:
                box["type"] = block_type
                box["coordinates"] = box["text_coordinates"]
                box["tei"] = box["text_content"]
                boxes.append(box)
    return boxes

def introduce_lb(boxes):
    for box in boxes:
        if box["is_line_break"]:
            box["tei"] += "<lb/>"
    return boxes

In [403]:
class Song:
    def __init__(self, file_path: str):
        with open(file_path, "r") as file_handle:
            self.json = json.load(file_handle)
            self.boxes = self.json["content"]

        self.image_files = self.json["images"]
        self.file = os.path.basename(file_path)
        self.title = "".join([box["text_content"] for box in self.boxes if box["box_type"] == "Title"])

    def __str__(self):
        return f"Song({self.title}, {self.file})"
    
    def to_tei(self):
        print([box["box_type"] for box in self.boxes])
        blocks = segment_into_blocks(self.boxes)
        print(block[0]["box_type"] for block in blocks)
        blocks = add_block_tags(blocks)
        boxes = resolve_blocks_to_boxes(blocks)
        boxes = introduce_lb(boxes)
        #introduce_pb(images, boxes)

        tei_string = ""
        for box in boxes:
            tei_string += box["tei"]

        return f'<div type="song">{tei_string}</div>'

class Table:
    def __init__(self, file_path: str):
        pass


class Description:
    def __init__(self, file_path: str):
        pass


class Section:
    def __init__(self, information_file: str, children: list[Description | Table | Song]):
        with open(information_file, "r") as file_handle:
            self.information_json = json.load(file_handle)
            self.information_boxes = self.information_json["content"]
            self.information_boxes = [box for box in self.information_boxes if box["box_type"] == "Title"]

        self.children = children
        self.image_files = set(self.information_json["images"] +
                               [file for child in self.children for file in child.image_files])
        self.boxes = (self.information_boxes + [box for child in self.children for box in child.boxes])
        self.file = os.path.basename(information_file)
        self.title = "".join([box["text_content"] for box in self.information_boxes])
        
    def __iter__(self):
        return iter(self.children)
    
    def __getitem__(self, idx: int):
        return self.children[idx]
        
    def __str__(self):
        return_string = f"Section({self.title}, {self.file})\n"
        for child in self.children:
            return_string += f"    {child}\n"
        return_string = return_string[:-1]
        return return_string
    
    def to_tei(self):
        tei_string = ""
        for child in self.children:
            tei_string += child.to_tei()
        return f'<div type="section"><head>{self.title}</head>{tei_string}</div>'

    
class Juan:
    def __init__(self, information_file: str, children: list[Section]):
        with open(information_file, "r") as file_handle:
            self.information_json = json.load(file_handle)
            self.information_boxes = self.information_json["content"]
            self.information_boxes = [box for box in self.information_boxes if box["box_type"] == "Unmarked"]

        self.children = children
        self.image_files = set(self.information_json["images"] +
                               [file for child in self.children for file in child.image_files])
        self.boxes = (self.information_boxes + [box for child in self.children for box in child.boxes])
        self.title = "".join([box["text_content"] for box in self.information_boxes])
        
    def __iter__(self):
        return iter(self.children)
    
    def __getitem__(self, idx: int):
        return self.children[idx]
        
    def __str__(self):
        return_string = f"Juan({self.title})\n"
        for child in self.children:
            child_string = str(child).split("\n")
            child_string = "".join(["    "+string+"\n" for string in child_string])
            return_string += f"{child_string}"
        return_string = return_string[:-1]
        return return_string
    
    def to_tei(self):
        tei_string = ""
        for child in self.children:
            tei_string += child.to_tei()
        return f'<div type="juan"><head>{self.title}</head>{tei_string}</div>'

class TableOfContents:
    def __init__(self, file_path: str):
        pass


class Book:
    def __init__(self, children: list[TableOfContents | Juan | Section]):
        self.children = children
        self.image_files = set([file for child in self.children for file in child.image_files])
        self.boxes = [box for child in self.children for box in child.boxes]
        
    def __iter__(self):
        return iter(self.children)
    
    def __getitem__(self, idx: int):
        return self.children[idx]
        
    def __str__(self):
        return_string = f"Book(\n"
        for child in self.children:
            child_string = str(child).split("\n")
            child_string = "".join(["    "+string+"\n" for string in child_string])
            return_string += f"{child_string}"
        return_string = return_string[:-1] + "\n)"
        return return_string
    
    def to_tei(self):
        tei_string = ""
        for child in self.children:
            tei_string += child.to_tei()
        return f'<div type="book">{tei_string}</div>'
        

In [411]:
book = Book([
    Juan(
        information_file='./json/01_shengsongnaogeguchuiqushisishou/shengsongnaogeguchuiqushisishou.json',
        children=[
            Section(
                information_file='./json/01_shengsongnaogeguchuiqushisishou/shengsongnaogeguchuiqushisishou.json',
                children=[
                    Song('./json/01_shengsongnaogeguchuiqushisishou/01_shangdiming.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/02_hezhibiao.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/03_huaihaizhuo.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/04_yuanzhishang.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/05_huangweichang.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/06_shushansui.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/07_shiyupei.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/08_wangzhongshan.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/09_dazairen.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/10_ougegui.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/11_fagongji.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/12_dilinyong.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/13_weisiye.json'),
                    Song('./json/01_shengsongnaogeguchuiqushisishou/14_yanjingfu.json'),
                ]),
            Section(
                information_file='./json/02_qinquyishou/qinquyishou.json',
                children=[
                    #Description('./json/02_qinquyishou/ceshangdiao.json'),
                    #Table('./json/02_qinquyishou/diaoxianfa.json'),
                    Song('./json/02_qinquyishou/015_guyuan.json'),
                ])
        ]),
    Juan(
        information_file='./json/03_yuejiuge/yuejiuge.json',
        children=[
            Section(
                information_file='./json/03_yuejiuge/yuejiuge.json',
                children=[
                    Song('./json/03_yuejiuge/016_dishunchudiao.json'),
                    Song('./json/03_yuejiuge/017_wangyuwudiao.json'),
                    Song('./json/03_yuejiuge/018_yuewangyuediao.json'),
                    Song('./json/03_yuejiuge/019_yuexiangceshangdiao.json'),
                    Song('./json/03_yuejiuge/020_xiangwanggupingdiao.json'),
                    Song('./json/03_yuejiuge/021_taozhishenshuangdiao.json'),
                    Song('./json/03_yuejiuge/022_caoeshucediao.json'),
                    Song('./json/03_yuejiuge/023_pangjiangjungaopingdiao.json'),
                    Song('./json/03_yuejiuge/024_jingzhongzhongguanshangdiao.json'),
                    Song('./json/03_yuejiuge/025_caixiaozizhongguanbanzhandiao.json'),
                    #Table('./json/03_yuejiuge/gujinpufa.json'),
                    #Description('./json/03_yuejiuge/zhezifa.json'),
                ]),
        ]),
    Juan(
        information_file='./json/04_ling/ling.json',
        children=[
            Section(
                information_file='./json/04_ling/ling.json',
                children=[
                    Song('./json/04_ling/026_xiaochongshanling.json'),
                    Song('./json/04_ling/027_jiangmeiyin.json'),
                    Song('./json/04_ling/028_moshanxi.json'),
                    Song('./json/04_ling/029_yingshengraohonglou.json'),
                    Song('./json/04_ling/030_geximeiling.json'),
                    Song('./json/04_ling/031_ruanlanggui.json'),
                    Song('./json/04_ling/032_ruanlanggui.json'),
                    Song('./json/04_ling/033_haoshijin.json'),
                    Song('./json/04_ling/034_dianjiangchun.json'),
                    Song('./json/04_ling/035_dianjiangchun.json'),
                    Song('./json/04_ling/036_yumeiren.json'),
                    Song('./json/04_ling/037_yumeiren.json'),
                    Song('./json/04_ling/038_yiwangsun.json'),
                    Song('./json/04_ling/039_shaonianyou.json'),
                    Song('./json/04_ling/040_zhegutian.json'),
                    Song('./json/04_ling/041_zhegutian.json'),
                    Song('./json/04_ling/042_zhegutian.json'),
                    Song('./json/04_ling/043_zhegutian.json'),
                    Song('./json/04_ling/044_zhegutian.json'),
                    Song('./json/04_ling/045_zhegutian.json'),
                    Song('./json/04_ling/046_zhegutian.json'),
                    Song('./json/04_ling/047_yexingchuan.json'),
                    Song('./json/04_ling/048_xinghuatianying.json'),
                    Song('./json/04_ling/049_zuiyinshangxiaopin.json'),
                    Song('./json/04_ling/050_yumeiling.json'),
                    Song('./json/04_ling/051_tashaxing.json'),
                    Song('./json/04_ling/052_suzhongqing.json'),
                    Song('./json/04_ling/053_huanxisha.json'),
                    Song('./json/04_ling/054_huanxisha.json'),
                    Song('./json/04_ling/055_huanxisha.json'),
                    Song('./json/04_ling/056_huanxisha.json'),
                    Song('./json/04_ling/057_huanxisha.json'),
                    Song('./json/04_ling/058_huanxisha.json'),
                ]),
        ]),
    Juan(
        information_file='./json/05_man/man.json',
        children=[
            Section(
                information_file='./json/05_man/man.json',
                children=[
                    Song('./json/05_man/059_nishangzhongxu.json'),
                    Song('./json/05_man/060_qinggongchun.json'),
                    Song('./json/05_man/061_qitianle.json'),
                    Song('./json/05_man/062_manjianghong.json'),
                    Song('./json/05_man/063_yiehong.json'),
                    Song('./json/05_man/064_niannujiao.json'),
                    Song('./json/05_man/065_niannujiao.json'),
                    Song('./json/05_man/066_meiwu.json'),
                    Song('./json/05_man/067_yuexiadi.json'),
                    Song('./json/05_man/068_qingboyin.json'),
                    Song('./json/05_man/069_faquxianxianyin.json'),
                    Song('./json/05_man/070_pipaxian.json'),
                    Song('./json/05_man/071_linglongsifan.json'),
                    Song('./json/05_man/072_cefan.json'),
                    Song('./json/05_man/073_shuilongyin.json'),
                    Song('./json/05_man/074_tanchunman.json'),
                    Song('./json/05_man/075_bagui.json'),
                    Song('./json/05_man/076_jielianhuan.json'),
                    Song('./json/05_man/077_xiyingqianman.json'),
                    Song('./json/05_man/078_moyuer.json'),
                ]),
        ]),
    Juan(
        information_file='./json/06_ziduqu/ziduqu.json',
        children=[
            Section(
                information_file='./json/06_ziduqu/ziduqu.json',
                children=[
                    Song('./json/06_ziduqu/079_yangzhouman.json'),
                    Song('./json/06_ziduqu/080_changtingyuanman.json'),
                    Song('./json/06_ziduqu/081_danhuangliu.json'),
                    Song('./json/06_ziduqu/082_shihuxian.json'),
                    Song('./json/06_ziduqu/083_anxiang.json'),
                    Song('./json/06_ziduqu/084_shuying.json'),
                    Song('./json/06_ziduqu/085_xihongyi.json'),
                    Song('./json/06_ziduqu/086_jueshao.json'),
                    Song('./json/06_ziduqu/087_zhishao.json'),
                ]),
        ]),
    Juan(
        information_file='./json/07_zizhiqu/zizhiqu.json',
        children=[
            Section(
                information_file='./json/07_zizhiqu/zizhiqu.json',
                children=[
                    Song('./json/07_zizhiqu/088_quixiaoyin.json'),
                    Song('./json/07_zizhiqu/089_qiliangfan.json'),
                    Song('./json/07_zizhiqu/090_cuilouyin.json'),
                    Song('./json/07_zizhiqu/091_xiangyue.json'),
                    Song('./json/07_zizhiqu/qingyuanhuiyao.json'),
                ]),
        ]),
    Juan(
        information_file='./json/08_bieji/bieji.json',
        children=[
            Section(
                information_file='./json/08_bieji/bieji.json',
                children=[
                    Song('./json/07_zizhiqu/088_quixiaoyin.json'),
                    Song('./json/07_zizhiqu/089_qiliangfan.json'),
                    Song('./json/07_zizhiqu/090_cuilouyin.json'),
                    Song('./json/07_zizhiqu/091_xiangyue.json'),
                    Song('./json/07_zizhiqu/qingyuanhuiyao.json'),
                ]),
        ]),])

In [412]:
print(book[1][0][0].to_tei())

['Unmarked', 'Title', 'Title', 'Title', 'Title', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music', 'Music']
<generator object Song.to_tei.<locals>.<genexpr> at 0x73d36577d080>
<div type="song">右<head>帝舜楚調</head><p ana="lyrics"><lg><l>央央帝旂群冕相輿聿來我媯我芸緑滋 <lb/>維湘與楚謂狩在陼雲横九疑帝若來下 <lb/>我懷厥初孰畊孰漁勿忘惠康疇匪帝餘 <lb/>博碩于爼維錯于豆瑶灑玉離侑此桂酒</l></lg></p><lb/></div>


In [172]:
def transform_juan_to_tei(juan_path: str):
    tei_string = ""
    all_jsons = sorted(os.listdir(juan_path))
    
    pieces = [json for json in all_jsons if json[0].isnumeric()]
    others = [json for json in all_jsons if not json[0].isnumeric()]
    
    for other in others:
        tei_string += transform_juan_description_to_tei(os.path.join(juan, other))
            
    for piece in pieces:
        tei_string += transform_piece_to_tei(os.path.join(juan, piece))
            
    return f'<div type="juan">{tei_string}</div>'

In [ ]:
def transform_book_to_tei(juan_division: list[str], other: str):
    tei_string = ""
    
    # Table of Contents
    # Juan 1: 01_shengsongnaogeguchuiqushisishou, 02_qinquyishou
    tei_string 
    # Juan 2: 03_yuejiuge
    # Juan 3: 04_ling
    # Juan 4: 05_man
    # Juan 5: 06_ziduqu
    # Juan 6: 07_zizhiqu
    # Appendix: 08_bieji
        
    return f'<div type="book">{tei_string}</div>'
    #return f'<div type="book" style="writing-mode: vertical-rl">{tei_string}</div>'

In [208]:
base_directory = "./json"

all_files = []
for folder in sorted(next(os.walk(base_directory))[1]):
    everything = sorted(os.listdir(os.path.join(base_directory, folder)))
    everything = [os.path.join(base_directory, folder, e) for e in everything]
    pieces = [e for e in everything if e[0].isnumeric()]
    other = [e for e in everything if not e[0].isnumeric()]
    
    all_files += other
    all_files += pieces
    
print(all_files)


['./json/01_shengsongnaogeguchuiqushisishou/01_shangdiming.json', './json/01_shengsongnaogeguchuiqushisishou/02_hezhibiao.json', './json/01_shengsongnaogeguchuiqushisishou/03_huaihaizhuo.json', './json/01_shengsongnaogeguchuiqushisishou/04_yuanzhishang.json', './json/01_shengsongnaogeguchuiqushisishou/05_huangweichang.json', './json/01_shengsongnaogeguchuiqushisishou/06_shushansui.json', './json/01_shengsongnaogeguchuiqushisishou/07_shiyupei.json', './json/01_shengsongnaogeguchuiqushisishou/08_wangzhongshan.json', './json/01_shengsongnaogeguchuiqushisishou/09_dazairen.json', './json/01_shengsongnaogeguchuiqushisishou/10_ougegui.json', './json/01_shengsongnaogeguchuiqushisishou/11_fagongji.json', './json/01_shengsongnaogeguchuiqushisishou/12_dilinyong.json', './json/01_shengsongnaogeguchuiqushisishou/13_weisiye.json', './json/01_shengsongnaogeguchuiqushisishou/14_yanjingfu.json', './json/01_shengsongnaogeguchuiqushisishou/shengsongnaogeguchuiqushisishou.json', './json/02_qinquyishou/015